# Clasificacion con Spotify Tracks Dataset

En este notebook vas a construir modelos de clasificacion para predecir caracteristicas de canciones usando audio features de Spotify.

El dataset contiene 114,000 canciones con las siguientes columnas relevantes:

| Columna | Descripcion |
|---|---|
| `popularity` | Popularidad de 0 a 100 |
| `duration_ms` | Duracion en milisegundos |
| `danceability` | Que tan bailable es la cancion (0-1) |
| `energy` | Intensidad y actividad percibida (0-1) |
| `loudness` | Volumen medio en dB |
| `speechiness` | Presencia de palabras habladas (0-1) |
| `acousticness` | Probabilidad de ser acustica (0-1) |
| `instrumentalness` | Probabilidad de no tener vocals (0-1) |
| `liveness` | Probabilidad de ser grabacion en vivo (0-1) |
| `valence` | Positividad musical (0-1) |
| `tempo` | Velocidad en BPM |
| `explicit` | Si la cancion tiene contenido explicito (True/False) |
| `track_genre` | Genero musical (114 categorias) |

**Dos ejercicios:**
- **Parte A — Clasificacion binaria:** predecir si una cancion es `explicit`
- **Parte B — Clasificacion multiclase:** predecir el genero entre `pop`, `rock`, `hip-hop` y `classical`

---

## 0. Librerias

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler

## 1. Carga y exploracion

Carga el archivo `dataset.csv` y revisa su estructura.

Resultado esperado:
```
Shape: (114000, 21)

Nulos por columna:
artists       1
album_name    1
track_name    1
(resto de columnas: 0)
```

In [ ]:
# Carga el dataset
# Imprime shape
# Imprime los nulos por columna


## 2. Limpieza

Elimina las filas con valores nulos. Son solo 3 filas sobre 114,000 — no hay riesgo de perder informacion relevante.

Resultado esperado:
```
Shape tras limpieza: (113999, 21)
```

In [ ]:
# Elimina filas con nulos
# Imprime el shape resultante


## 3. Analisis exploratorio

Antes de modelar, entiende las dos variables objetivo.

1. Distribucion de `explicit` (cuantas canciones son y no son explicitas)
2. Histograma de las features numericas principales
3. Boxplot de `danceability`, `energy` y `speechiness` agrupados por `explicit`

Resultado esperado para la distribucion:
```
explicit
False    104252
True      9747
```

In [ ]:
# Distribucion de explicit


In [ ]:
# Histogramas de features numericas


In [ ]:
# Boxplot de danceability, energy y speechiness agrupados por explicit


---
# Parte A — Clasificacion Binaria
**Objetivo: predecir si una cancion es `explicit`**

---

## 4. Preparacion de datos (binaria)

Usa estas 11 features para predecir `explicit`:
`popularity`, `duration_ms`, `danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`

- Convierte `explicit` a entero (0/1)
- Split 80/20 con `random_state=42` y `stratify=y` (importante por el desbalance)
- Estandariza con `StandardScaler`

Resultado esperado:
```
Clase 0 (no explicit): 104252
Clase 1 (explicit):      9747

X_train: (91199, 11)   X_test: (22800, 11)
```

In [ ]:
# Define feature_cols y el target y
# Imprime la distribucion de clases
# Haz el split con stratify
# Estandariza


## 5. Modelo binario

Entrena una `LogisticRegression` con `max_iter=1000` y `random_state=42`.

Resultado esperado:
```
Accuracy: 0.9143

              precision    recall  f1-score   support
           0       0.92      0.99      0.95     20851
           1       0.49      0.09      0.16      1949
    accuracy                           0.91     22800
```

In [ ]:
# Entrena LogisticRegression
# Imprime accuracy y classification_report


## 6. Matriz de confusion (binaria)

Visualiza la matriz de confusion con `ConfusionMatrixDisplay`.

Resultado esperado:
```
[[20666   185]
 [ 1769   180]]
```

In [ ]:
# Visualiza la matriz de confusion


## 7. Analisis del desbalance

El accuracy de 91% parece bueno, pero mira el recall de la clase 1 (explicit): solo 0.09. El modelo casi nunca acierta canciones explicitas.

Esto es un problema de **clases desbalanceadas**. Una solucion simple es usar el parametro `class_weight='balanced'` en `LogisticRegression`, que penaliza mas los errores en la clase minoritaria.

Reentrena el modelo con `class_weight='balanced'` y compara los resultados.

Resultado esperado:
```
              precision    recall  f1-score   support
           0       0.95      0.80      0.87     20851
           1       0.35      0.73      0.47      1949
    accuracy                           0.80     22800
```

In [ ]:
# Reentrena con class_weight='balanced'
# Imprime accuracy y classification_report
# Visualiza la nueva matriz de confusion


## 8. Coeficientes del modelo

Visualiza los coeficientes del modelo balanceado en un grafico de barras horizontal ordenado de mayor a menor.

Resultado esperado (coeficientes ordenados):
```
speechiness        0.6452
danceability       0.4974
loudness           0.4732
popularity         0.1658
tempo             -0.0362
liveness          -0.0565
energy            -0.0715
duration_ms       -0.3667
instrumentalness  -0.4253
valence           -0.4300
acousticness      -0.4923
```

In [ ]:
# Grafico de barras horizontal con los coeficientes


---
# Parte B — Clasificacion Multiclase
**Objetivo: predecir el genero entre `pop`, `rock`, `hip-hop` y `classical`**

---

## 9. Preparacion de datos (multiclase)

Filtra el dataset para quedarte solo con los 4 generos: `pop`, `rock`, `hip-hop`, `classical`. Usa las mismas 11 features.

- Split 80/20 con `random_state=42` y `stratify=y`
- Estandariza con un nuevo `StandardScaler`

Resultado esperado:
```
Distribucion por genero:
classical    1000
hip-hop      1000
pop          1000
rock         1000

X_train: (3200, 11)   X_test: (800, 11)
```

In [ ]:
# Filtra el dataset a los 4 generos
# Imprime la distribucion
# Split y estandarizacion


## 10. Modelo multiclase

Entrena una `LogisticRegression` con `max_iter=1000` y `random_state=42`. Sklearn detecta automaticamente que hay mas de 2 clases y usa clasificacion multiclase.

Resultado esperado:
```
Accuracy: 0.7075

              precision    recall  f1-score   support
   classical       0.93      0.93      0.93       200
     hip-hop       0.71      0.70      0.71       200
         pop       0.53      0.49      0.51       200
        rock       0.66      0.70      0.68       200
    accuracy                           0.71       800
```

In [ ]:
# Entrena LogisticRegression
# Imprime accuracy y classification_report


## 11. Matriz de confusion (multiclase)

Visualiza la matriz de confusion. Con 4 clases puedes ver exactamente que pares de generos se confunden mas.

Resultado esperado:
```
              pop   rock  hip-hop  classical
pop          [ 99    43      47       11]
rock         [ 45   141      10        4]
hip-hop      [ 33    26     141        0]
classical    [ 11     4       0      185]
```

In [ ]:
# Visualiza la matriz de confusion con ConfusionMatrixDisplay


## 12. Reflexion final

Responde en esta celda:

1. El accuracy del modelo binario sin balancear es 91%, pero el modelo balanceado baja a 80%. Cual es mejor y por que depende del contexto?
2. Por que `speechiness` tiene el coeficiente positivo mas alto para predecir `explicit`? Tiene sentido musicalmente?
3. `classical` se clasifica con precision y recall de 0.93, mientras `pop` solo llega a 0.53. Por que crees que classical es tan facil de separar y pop tan dificil?
4. Que generos se confunden mas entre si segun la matriz de confusion? Por que tiene sentido?
5. El baseline de la clasificacion multiclase es 0.25 (adivinar al azar entre 4 clases). El modelo llega a 0.71. Es un resultado razonable con solo features de audio?

*Escribe tu respuesta aqui*